# Galfit

`Galfit_Fitter` fits Sersic and/or point-source profiles to galaxy
cutouts using the GALFIT code (Peng et al. 2002/2010): box-constrained
parameters written to a text input file, then fit by shelling out to
the `galfit` binary. It shares the same `Morphology_Fitter` interface
as [`PySersic_Fitter`](PySersic.ipynb) -- `primary_constraints`/
`fid_params` play the same "fixed value or `[lo, hi]` range" role that
`PySersic_Fitter`'s `primary_priors` does, just as GALFIT box
constraints rather than Bayesian priors.

.. note::
   Requires a working GALFIT installation (the `galfit` binary on
   `$PATH`, or `GALFIT_INSTALL_PATH` set in the galfind config).

This notebook fits the same real source as the PySersic notebook --
GS-z14-1 (Carniani+24, JADES), a spectroscopically-confirmed z~14.3
galaxy -- in NIRCam/F444W, with `n` held fixed at 1.

In [ ]:
import sys
sys.path.append("/nvme/scratch/work/austind/EPOCHS-v2/scripts")

import astropy.units as u
from galfind import config as galfind_config
from galfind.galaxy import Galaxy
from galfind.selection.Selector import ID_Selector
from galfind.properties.Morphology import Galfit_Fitter
from config import min_flux_pc_err
from load_cat import base_cat_load

survey = "JADES-DR3-GS-South"  # GS-z14-1 (Carniani+24)
version = "v13"
ID = 30479

# Crop to just this one source so the (real, heavy) forced-photometry /
# SExtractor pipeline only has to build a single Galaxy object -- used
# purely to load the SExtractor-derived quantities GALFIT needs
# (MAG_AUTO, Re, Kron radius/axes). The `Galaxy` actually fitted below
# is built explicitly via the `Galaxy.from_data_id` constructor, not by
# indexing this cropped catalogue.
sex_cat = base_cat_load(survey, version, crops=ID_Selector([ID]), load_sex_params=True)
band_data = sex_cat.data.native["F444W"]

gal = Galaxy.from_data_id(
    sex_cat.data, ID,
    load_phot_kwargs={"ZP": u.Jy.to(u.ABmag), "min_flux_pc_err": min_flux_pc_err},
)
for attr in (
    "sex_MAG_AUTO", "sex_Re", "sex_KRON_RADIUS", "sex_A_IMAGE", "sex_B_IMAGE",
    "sex_THETA_IMAGE", "sex_A_IMAGE_AS", "sex_B_IMAGE_AS",
):
    setattr(gal, attr, getattr(sex_cat[0], attr))

print(gal, gal.sex_MAG_AUTO["F444W"], gal.sex_Re["F444W"])


`Catalogue.load_sextractor_kron_radii` computes `sex_A_IMAGE_AS`/
`sex_B_IMAGE_AS` (needed by `Galfit_Fitter._fit_cutout`'s
`cutout.meta["A_IMAGE_AS"]`/`"B_IMAGE_AS"`) as part of
`load_sextractor_params()` above, so this is a no-op on a fresh
catalogue -- kept as a safety net in case a `Catalogue` is re-used
across multiple loads (`load_sextractor_kron_radii` skips recomputing
anything once these are already set). Note `sex_A_IMAGE`/`sex_B_IMAGE`
are plain scalars (one representative band), not per-band dicts like
`sex_KRON_RADIUS` -- see `Galaxy.plot`'s identical usage.

In [ ]:
if not hasattr(gal, "sex_A_IMAGE_AS"):
    gal.sex_A_IMAGE_AS = {
        "F444W": gal.sex_KRON_RADIUS["F444W"] * gal.sex_A_IMAGE * band_data.pix_scale
    }
    gal.sex_B_IMAGE_AS = {
        "F444W": gal.sex_KRON_RADIUS["F444W"] * gal.sex_B_IMAGE * band_data.pix_scale
    }

cutout = gal.make_band_cutout(band_data, cutout_size=1.5 * u.arcsec, overwrite=True)

fitter = Galfit_Fitter(
    psf=band_data.psf, model="sersic",
    primary_constraints={"sersic": {"x": [-2, 2], "y": [-2, 2], "n": 1.0}},
    neighbours_model=None,
    fid_params={"sersic": {"n": 1, "axr": 1, "pa": 0}},
)
# Saves/loads under GALFIND_WORK/GALFIT, following the same per-source
# directory convention `Galfit_Fitter._fit_cat` builds by default --
# re-running this cell reloads the cached GALFIT output rather than
# re-fitting.
morph_dirs = f"{version}/{survey}/F444W/{ID}"
result = fitter._fit_cutout(
    cutout,
    fid_mag=gal.sex_MAG_AUTO["F444W"],
    fid_re=gal.sex_Re["F444W"],
    in_dir=f"{galfind_config['GALFIT']['INPUT_DIR']}/{morph_dirs}",
    out_dir=f"{galfind_config['GALFIT']['OUTPUT_DIR']}/{morph_dirs}",
    cat=None,
    plot=True,
)
print(result.properties if result is not None else "fit failed/crashed")


### Image / model / residual

`Galfit_Result.plot()` shows the original science image, GALFIT's
best-fit model, and the residual, with the primary source's Kron
aperture overlaid on the residual panel and masked pixels contoured in
red.

In [ ]:
result.plot(save=False, show=True)
